# P=1 QAOA Energy Landscapes

Comparing the expected cost landscape over (γ, β) for:
1. **Penalty QAOA** — uniform initial state, product-RX mixer
2. **XY Mixer QAOA** — weight-s initial state, ring XY mixer

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time
import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)

from pde_qaoa_helpers import (
    load_pde_data, generate_pde_qubo, compute_lambda,
    build_penalty_matrices, build_bqo_matrices,
    precompute_costs,
    landscape_penalty, landscape_xy_mixer,
)

## Problem Setup

In [ ]:
Phi, M, Kinv, yhat_vec = load_pde_data('ifiss3.6/QUBO_3.mat')
s = 2

Q, q, c = generate_pde_qubo(M, Kinv, Phi, yhat_vec)
n_original = q.shape[0]
print(f'Q: {Q.shape}, q: {q.shape}, c: {float(c):.6f}')
print(f'Number of original decision variables: {n_original}')

In [ ]:
lam_tight = compute_lambda(Q, q, c, s)
pen_Q, pen_q, pen_c, n_pen = build_penalty_matrices(Q, q, c, s, lam_tight)
xy_Q, xy_q, xy_c, n_xy = build_bqo_matrices(Q, q, c, s)

print(f'Tight penalty lambda: {lam_tight:.4f}')
print(f'Penalty QUBO: {n_pen} qubits')
print(f'XY Mixer BQO: {n_xy} qubits')

## Precompute Costs

In [ ]:
t0 = time.time()
pen_costs = precompute_costs(pen_Q, pen_q, pen_c, n_pen)
t1 = time.time()
print(f'Penalty costs ({n_pen} qubits, {2**n_pen} states): {t1-t0:.2f}s')

xy_costs = precompute_costs(xy_Q, xy_q, xy_c, n_xy)
t2 = time.time()
print(f'XY Mixer costs ({n_xy} qubits, {2**n_xy} states): {t2-t1:.2f}s')

## Compute Landscapes

In [ ]:
N_GRID = 100

pen_gamma_range = np.linspace(0, 2 * np.pi, N_GRID)
xy_gamma_range = np.linspace(0, 2 * np.pi, N_GRID)
beta_range = np.linspace(0, np.pi, N_GRID)

print(f'Grid: {N_GRID}x{N_GRID} = {N_GRID**2} points per landscape\n')

t0 = time.time()
pen_landscape = landscape_penalty(pen_costs, n_pen, pen_gamma_range, beta_range)
t1 = time.time()
print(f'Penalty landscape ({n_pen} qubits): {t1-t0:.1f}s')

xy_landscape = landscape_xy_mixer(xy_costs, n_xy, s, xy_gamma_range, beta_range)
t2 = time.time()
print(f'XY Mixer landscape ({n_xy} qubits): {t2-t1:.1f}s')

## Visualize

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5), dpi=150)

im1 = ax1.pcolormesh(pen_gamma_range, beta_range, pen_landscape,
                      shading='auto', cmap='viridis')
pen_min = np.unravel_index(np.argmin(pen_landscape), pen_landscape.shape)
ax1.plot(pen_gamma_range[pen_min[1]], beta_range[pen_min[0]], 'r*',
         markersize=14, label='Grid minimum')
ax1.plot(0, 0, 'wx', markersize=10, markeredgewidth=2, label='Initial point (0, 0)')
ax1.set_xlabel(r'$\gamma$', fontsize=12)
ax1.set_ylabel(r'$\beta$', fontsize=12)
ax1.set_title(f'Penalty QAOA (P=1)\n$\\lambda={lam_tight:.1f}$, {n_pen} qubits', fontsize=11)
ax1.legend(loc='upper right', fontsize=8)
fig.colorbar(im1, ax=ax1, label='Expected Cost')

im2 = ax2.pcolormesh(xy_gamma_range, beta_range, xy_landscape,
                      shading='auto', cmap='viridis')
xy_min = np.unravel_index(np.argmin(xy_landscape), xy_landscape.shape)
ax2.plot(xy_gamma_range[xy_min[1]], beta_range[xy_min[0]], 'r*',
         markersize=14, label='Grid minimum')
ax2.plot(0, 0, 'wx', markersize=10, markeredgewidth=2, label='Initial point (0, 0)')
ax2.set_xlabel(r'$\gamma$', fontsize=12)
ax2.set_ylabel(r'$\beta$', fontsize=12)
ax2.set_title(f'XY Mixer QAOA (P=1)\n{n_xy} qubits, Ring topology', fontsize=11)
ax2.legend(loc='upper right', fontsize=8)
fig.colorbar(im2, ax=ax2, label='Expected Cost')

plt.tight_layout()
plt.savefig('qaoa_p1_landscapes.png', bbox_inches='tight')
plt.show()

print(f'Penalty range: [{pen_landscape.min():.4f}, {pen_landscape.max():.4f}]')
print(f'  Min at gamma={pen_gamma_range[pen_min[1]]:.4f}, beta={beta_range[pen_min[0]]:.4f}')
print(f'XY Mixer range: [{xy_landscape.min():.4f}, {xy_landscape.max():.4f}]')
print(f'  Min at gamma={xy_gamma_range[xy_min[1]]:.4f}, beta={beta_range[xy_min[0]]:.4f}')